Cell 1: Imports

In [1]:
# Cell 1: Imports

import os
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

Cell 2: Config / Paths (full split version)

In [2]:
# Cell 2: Config / Paths (full split version)


import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

ROOT = Path.cwd().parents[1]
print("ROOT:", ROOT)

# ===== full cleaned data =====
CLEAN_DIR = ROOT / "data" / "processed" / "clean_chunks22"

# ===== canonical split =====
PREDS_DIR = ROOT / "results" / "preds"
SPLIT_TRAIN_SEQIDS_PATH = PREDS_DIR / "split_train_seq_ids.npy"
SPLIT_TEST_SEQIDS_PATH = PREDS_DIR / "split_test_seq_ids.npy"

# ===== class names =====
TOF_DIR = ROOT / "data" / "processed_data_tof"
CLASSES_PATH = TOF_DIR / "gesture_classes_raw.npy"

print("CLEAN_DIR:", CLEAN_DIR)
print("SPLIT_TRAIN_SEQIDS_PATH:", SPLIT_TRAIN_SEQIDS_PATH)
print("SPLIT_TEST_SEQIDS_PATH:", SPLIT_TEST_SEQIDS_PATH)
print("CLASSES_PATH:", CLASSES_PATH)

DEVICE: cpu
ROOT: D:\0-about me\2-FDU\4-CSCI 6806 CAPSTONE\1_from github\2026-winter-capstone-project-2026winter-capstone-group-11
CLEAN_DIR: D:\0-about me\2-FDU\4-CSCI 6806 CAPSTONE\1_from github\2026-winter-capstone-project-2026winter-capstone-group-11\data\processed\clean_chunks22
SPLIT_TRAIN_SEQIDS_PATH: D:\0-about me\2-FDU\4-CSCI 6806 CAPSTONE\1_from github\2026-winter-capstone-project-2026winter-capstone-group-11\results\preds\split_train_seq_ids.npy
SPLIT_TEST_SEQIDS_PATH: D:\0-about me\2-FDU\4-CSCI 6806 CAPSTONE\1_from github\2026-winter-capstone-project-2026winter-capstone-group-11\results\preds\split_test_seq_ids.npy
CLASSES_PATH: D:\0-about me\2-FDU\4-CSCI 6806 CAPSTONE\1_from github\2026-winter-capstone-project-2026winter-capstone-group-11\data\processed_data_tof\gesture_classes_raw.npy


Cell 3: Load clean full data + canonical split

In [3]:
# Cell 3: Load clean full data + canonical split

clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
print("clean files:", clean_files)

dfs = []
for fp in clean_files:
    df_part = pd.read_parquet(fp)
    print(fp.name, df_part.shape)
    dfs.append(df_part)

full_df = pd.concat(dfs, axis=0, ignore_index=True)

train_seqids = np.load(SPLIT_TRAIN_SEQIDS_PATH, allow_pickle=True)
test_seqids = np.load(SPLIT_TEST_SEQIDS_PATH, allow_pickle=True)

classes = np.load(CLASSES_PATH, allow_pickle=True)

print("full_df shape:", full_df.shape)
print("len(train_seqids):", len(train_seqids))
print("len(test_seqids):", len(test_seqids))
print("classes:", classes)

full_df shape: (536305, 341)
len(train_seqids): 6070
len(test_seqids): 1518
classes: ['Above ear - pull hair' 'Cheek - pinch skin' 'Drink from bottle/cup'
 'Eyebrow - pull hair' 'Eyelash - pull hair'
 'Feel around in tray and pull out an object' 'Forehead - pull hairline'
 'Forehead - scratch' 'Glasses on/off' 'Neck - pinch skin'
 'Neck - scratch' 'Pinch knee/leg skin' 'Pull air toward your face'
 'Scratch knee/leg skin' 'Text on phone' 'Wave hello' 'Write name in air'
 'Write name on leg']


Cell 4: Inspect columns

In [4]:
# Cell 4: Inspect columns

print("full_df columns:")
print(full_df.columns.tolist())

tof_cols = [c for c in full_df.columns if str(c).startswith("tof_")]
print("\nNumber of TOF columns:", len(tof_cols))
print("First 10 TOF cols:", tof_cols[:10])

candidate_seq_cols = ["sequence_id", "seq_id", "sequence_counter"]
candidate_label_cols = ["gesture", "label", "target", "behavior"]

seq_col = None
label_col = None

for c in candidate_seq_cols:
    if c in full_df.columns:
        seq_col = c
        break

for c in candidate_label_cols:
    if c in full_df.columns:
        label_col = c
        break

print("\nDetected seq_col:", seq_col)
print("Detected label_col:", label_col)

assert seq_col is not None, "No sequence ID column found."
assert label_col is not None, "No label column found."
assert len(tof_cols) == 320, f"Expected 320 TOF cols, got {len(tof_cols)}"

full_df columns:
['row_id', 'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5', 'tof_1_v0', 'tof_1_v1', 'tof_1_v2', 'tof_1_v3', 'tof_1_v4', 'tof_1_v5', 'tof_1_v6', 'tof_1_v7', 'tof_1_v8', 'tof_1_v9', 'tof_1_v10', 'tof_1_v11', 'tof_1_v12', 'tof_1_v13', 'tof_1_v14', 'tof_1_v15', 'tof_1_v16', 'tof_1_v17', 'tof_1_v18', 'tof_1_v19', 'tof_1_v20', 'tof_1_v21', 'tof_1_v22', 'tof_1_v23', 'tof_1_v24', 'tof_1_v25', 'tof_1_v26', 'tof_1_v27', 'tof_1_v28', 'tof_1_v29', 'tof_1_v30', 'tof_1_v31', 'tof_1_v32', 'tof_1_v33', 'tof_1_v34', 'tof_1_v35', 'tof_1_v36', 'tof_1_v37', 'tof_1_v38', 'tof_1_v39', 'tof_1_v40', 'tof_1_v41', 'tof_1_v42', 'tof_1_v43', 'tof_1_v44', 'tof_1_v45', 'tof_1_v46', 'tof_1_v47', 'tof_1_v48', 'tof_1_v49', 'tof_1_v50', 'tof_1_v51', 'tof_1_v52', 'tof_1_v53', 'tof_1_v54', 'tof_1_v55', 'tof_1_v56', 'tof_1_v57', 'tof_1_v58

Cell 5: Build full-split sequences + labels

In [5]:
# Cell 5: Build full-split sequences + labels

def build_sequence_list_and_labels(df, seq_ids, seq_col, feature_cols, label_col):
    seq_map = {}
    label_map = {}

    grouped = df.groupby(seq_col, sort=False)

    for sid, g in grouped:
        sid = str(sid)
        seq_map[sid] = g[feature_cols].to_numpy(dtype=np.float32)

        labels = g[label_col].astype(str).unique()
        assert len(labels) == 1, f"{sid} has multiple labels: {labels}"
        label_map[sid] = labels[0]

    sequences = []
    labels = []
    kept_seqids = []
    missing = []

    for sid in seq_ids:
        sid = str(sid)
        if sid in seq_map and sid in label_map:
            sequences.append(seq_map[sid])
            labels.append(label_map[sid])
            kept_seqids.append(sid)
        else:
            missing.append(sid)

    return sequences, labels, kept_seqids, missing


X_train_seq, y_train_str, seqid_train_used, missing_train = build_sequence_list_and_labels(
    full_df, train_seqids, seq_col, tof_cols, label_col
)

X_test_seq, y_test_str, seqid_test_used, missing_test = build_sequence_list_and_labels(
    full_df, test_seqids, seq_col, tof_cols, label_col
)

print("Train sequences built:", len(X_train_seq))
print("Test sequences built:", len(X_test_seq))
print("Missing train:", len(missing_train))
print("Missing test:", len(missing_test))

if len(X_train_seq) > 0:
    print("First train sequence shape:", X_train_seq[0].shape)
    print("First train seqid:", seqid_train_used[0])
    print("First train label:", y_train_str[0])

print("First 10 missing train:", missing_train[:10])
print("First 10 missing test:", missing_test[:10])

Train sequences built: 6070
Test sequences built: 1518
Missing train: 0
Missing test: 0
First train sequence shape: (68, 320)
First train seqid: SEQ_000008
First train label: Forehead - pull hairline
First 10 missing train: []
First 10 missing test: []


Cell 6: Convert labels to ids

In [6]:
# Cell 6: Convert labels to ids

label2id = {str(c): i for i, c in enumerate(classes)}
id2label = {i: str(c) for i, c in enumerate(classes)}

y_train = np.array([label2id[str(y)] for y in y_train_str], dtype=np.int64)
y_test = np.array([label2id[str(y)] for y in y_test_str], dtype=np.int64)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("First 10 y_train:", y_train[:10])

y_train shape: (6070,)
y_test shape: (1518,)
First 10 y_train: [ 6  1 17  6  5 10  9  6  3  0]


Cell 7: Inspect sequence lengths

In [7]:
# Cell 7: Inspect sequence lengths

train_lengths = np.array([seq.shape[0] for seq in X_train_seq], dtype=np.int32)
test_lengths = np.array([seq.shape[0] for seq in X_test_seq], dtype=np.int32)

print("Train length stats:")
print("min:", train_lengths.min())
print("max:", train_lengths.max())
print("mean:", train_lengths.mean())
print("median:", np.median(train_lengths))

print("\nTest length stats:")
print("min:", test_lengths.min())
print("max:", test_lengths.max())
print("mean:", test_lengths.mean())
print("median:", np.median(test_lengths))

Train length stats:
min: 35
max: 700
mean: 70.49126853377265
median: 59.0

Test length stats:
min: 35
max: 390
mean: 71.29973649538867
median: 59.0


Cell 8: Fit richer TOF tokenizer bins from training data

mean + std + min + max -> richer joint token

In [8]:
# Cell 8: Fit richer TOF tokenizer bins from training data
           #mean + std + min + max -> richer joint token

NUM_MEAN_BINS = 8
NUM_STD_BINS = 8
NUM_MIN_BINS = 8
NUM_MAX_BINS = 8

PAD_TOKEN_ID = 0  # 留给 padding

def frame_to_stats(seq_2d: np.ndarray):
    """
    seq_2d: shape (T, 320)
    return:
      frame_mean: shape (T,)
      frame_std:  shape (T,)
      frame_min:  shape (T,)
      frame_max:  shape (T,)
    """
    frame_mean = seq_2d.mean(axis=1)
    frame_std = seq_2d.std(axis=1)
    frame_min = seq_2d.min(axis=1)
    frame_max = seq_2d.max(axis=1)
    return frame_mean, frame_std, frame_min, frame_max

def fit_quantile_bins(values: np.ndarray, num_bins: int) -> np.ndarray:
    """
    Fit internal quantile bin edges.
    For num_bins bins, we need num_bins - 1 internal edges.
    """
    edges = np.quantile(
        values,
        q=np.linspace(0, 1, num_bins + 1)[1:-1]
    ).astype(np.float32)
    return edges

# 收集所有 training frames 的统计量，拿来拟合 bins
all_train_frame_means = []
all_train_frame_stds = []
all_train_frame_mins = []
all_train_frame_maxs = []

for seq in X_train_seq:
    frame_mean, frame_std, frame_min, frame_max = frame_to_stats(seq)
    all_train_frame_means.append(frame_mean)
    all_train_frame_stds.append(frame_std)
    all_train_frame_mins.append(frame_min)
    all_train_frame_maxs.append(frame_max)

all_train_frame_means = np.concatenate(all_train_frame_means, axis=0).astype(np.float32)
all_train_frame_stds = np.concatenate(all_train_frame_stds, axis=0).astype(np.float32)
all_train_frame_mins = np.concatenate(all_train_frame_mins, axis=0).astype(np.float32)
all_train_frame_maxs = np.concatenate(all_train_frame_maxs, axis=0).astype(np.float32)

print("Total train frames used to fit tokenizer:", len(all_train_frame_means))

print("\nFrame mean min/max:", all_train_frame_means.min(), all_train_frame_means.max())
print("Frame std  min/max:", all_train_frame_stds.min(), all_train_frame_stds.max())
print("Frame min  min/max:", all_train_frame_mins.min(), all_train_frame_mins.max())
print("Frame max  min/max:", all_train_frame_maxs.min(), all_train_frame_maxs.max())

mean_bin_edges = fit_quantile_bins(all_train_frame_means, NUM_MEAN_BINS)
std_bin_edges = fit_quantile_bins(all_train_frame_stds, NUM_STD_BINS)
min_bin_edges = fit_quantile_bins(all_train_frame_mins, NUM_MIN_BINS)
max_bin_edges = fit_quantile_bins(all_train_frame_maxs, NUM_MAX_BINS)

print("\nNumber of mean bin edges:", len(mean_bin_edges))
print("Number of std  bin edges:", len(std_bin_edges))
print("Number of min  bin edges:", len(min_bin_edges))
print("Number of max  bin edges:", len(max_bin_edges))

print("\nFirst 10 mean bin edges:", mean_bin_edges[:10])
print("First 10 std  bin edges:", std_bin_edges[:10])
print("First 10 min  bin edges:", min_bin_edges[:10])
print("First 10 max  bin edges:", max_bin_edges[:10])


Total train frames used to fit tokenizer: 427882

Frame mean min/max: -1.0 180.05937
Frame std  min/max: 0.0 116.77793
Frame min  min/max: -1.0 26.0
Frame max  min/max: -1.0 249.0

Number of mean bin edges: 7
Number of std  bin edges: 7
Number of min  bin edges: 7
Number of max  bin edges: 7

First 10 mean bin edges: [ 3.3625   13.846875 26.078125 39.38125  53.65625  70.18125  89.39375 ]
First 10 std  bin edges: [19.020184 34.75133  45.233204 54.54754  63.12422  71.67739  81.64581 ]
First 10 min  bin edges: [-1. -1. -1. -1. -1. -1. -1.]
First 10 max  bin edges: [128. 207. 234. 243. 246. 248. 249.]


Cell 9: Convert TOF sequences into richer token sequences

mean + std + min + max -> richer joint token

In [9]:
# Cell 9: Convert TOF sequences into richer token sequences
# Version 3: mean + std + min + max -> richer joint token

def sequence_to_token_ids(
    seq_2d: np.ndarray,
    mean_bin_edges: np.ndarray,
    std_bin_edges: np.ndarray,
    min_bin_edges: np.ndarray,
    max_bin_edges: np.ndarray,
) -> np.ndarray:
    """
    seq_2d: shape (T, 320)
    return: token ids, shape (T,)

    tokenization:
      1) discretize mean / std / min / max separately
      2) combine them into one joint token id
      3) +1 so that 0 is reserved for PAD

    token id range:
      1 ... (NUM_MEAN_BINS * NUM_STD_BINS * NUM_MIN_BINS * NUM_MAX_BINS)
      0 is reserved for PAD
    """
    frame_mean, frame_std, frame_min, frame_max = frame_to_stats(seq_2d)

    mean_ids = np.digitize(frame_mean, bins=mean_bin_edges, right=False)
    std_ids = np.digitize(frame_std, bins=std_bin_edges, right=False)
    min_ids = np.digitize(frame_min, bins=min_bin_edges, right=False)
    max_ids = np.digitize(frame_max, bins=max_bin_edges, right=False)

    joint_ids = (
        (((mean_ids * NUM_STD_BINS) + std_ids) * NUM_MIN_BINS + min_ids) * NUM_MAX_BINS
        + max_ids
    )

    token_ids = joint_ids + 1
    return token_ids.astype(np.int64)

train_token_seqs = [
    sequence_to_token_ids(seq, mean_bin_edges, std_bin_edges, min_bin_edges, max_bin_edges)
    for seq in X_train_seq
]
test_token_seqs = [
    sequence_to_token_ids(seq, mean_bin_edges, std_bin_edges, min_bin_edges, max_bin_edges)
    for seq in X_test_seq
]

print("Number of tokenized train sequences:", len(train_token_seqs))
print("First train token seq shape:", train_token_seqs[0].shape)
print("First 20 tokens of first train seq:", train_token_seqs[0][:20])

VOCAB_SIZE = (NUM_MEAN_BINS * NUM_STD_BINS * NUM_MIN_BINS * NUM_MAX_BINS) + 1
print("VOCAB_SIZE:", VOCAB_SIZE)


Number of tokenized train sequences: 6070
First train token seq shape: (68,)
First 20 tokens of first train seq: [1342 1344 1344 1408 1408 1407 1407 1407 1472 1472 1405 1470 1472 1983
 1472 1406 1472 1983 3070 3582]
VOCAB_SIZE: 4097


Cell 10: Pad / truncate token sequences

In [10]:
# Cell 10: Pad / truncate token sequences

MAX_LEN = 100

def pad_or_truncate(token_ids: np.ndarray, max_len: int, pad_value: int = PAD_TOKEN_ID):
    """
    token_ids: shape (T,)
    return:
      padded_ids: shape (max_len,)
      true_len: original effective length after truncation
    """
    T = len(token_ids)
    true_len = min(T, max_len)

    padded = np.full((max_len,), pad_value, dtype=np.int64)
    padded[:true_len] = token_ids[:true_len]
    return padded, true_len

X_train_tok = []
train_lens = []

for seq in train_token_seqs:
    padded, true_len = pad_or_truncate(seq, MAX_LEN)
    X_train_tok.append(padded)
    train_lens.append(true_len)

X_test_tok = []
test_lens = []

for seq in test_token_seqs:
    padded, true_len = pad_or_truncate(seq, MAX_LEN)
    X_test_tok.append(padded)
    test_lens.append(true_len)

X_train_tok = np.stack(X_train_tok)  # shape (N_train, MAX_LEN)
X_test_tok = np.stack(X_test_tok)    # shape (N_test, MAX_LEN)

train_lens = np.array(train_lens, dtype=np.int64)
test_lens = np.array(test_lens, dtype=np.int64)

print("X_train_tok shape:", X_train_tok.shape)
print("X_test_tok shape:", X_test_tok.shape)
print("train_lens shape:", train_lens.shape)
print("test_lens shape:", test_lens.shape)
print("First padded train token seq:", X_train_tok[0][:30])
print("First train true length:", train_lens[0])

X_train_tok shape: (6070, 100)
X_test_tok shape: (1518, 100)
train_lens shape: (6070,)
test_lens shape: (1518,)
First padded train token seq: [1342 1344 1344 1408 1408 1407 1407 1407 1472 1472 1405 1470 1472 1983
 1472 1406 1472 1983 3070 3582 3519 3451 3451 3387 3386 3386 3386 3387
 3387 3386]
First train true length: 68


Cell 11: Dataset / DataLoader (train / val / test)

In [11]:
# Cell 11: Dataset / DataLoader (train / val / test)

VAL_RATIO = 0.15
BATCH_SIZE = 64

train_indices, val_indices = train_test_split(
    np.arange(len(X_train_tok)),
    test_size=VAL_RATIO,
    random_state=SEED,
    stratify=y_train,
)

# Make seqids indexable by numpy integer arrays
seqid_train_used = np.array(seqid_train_used, dtype=object)
seqid_test_used = np.array(seqid_test_used, dtype=object)

X_subtrain_tok = X_train_tok[train_indices]
subtrain_lens = train_lens[train_indices]
y_subtrain = y_train[train_indices]
seqid_subtrain = seqid_train_used[train_indices]

X_val_tok = X_train_tok[val_indices]
val_lens = train_lens[val_indices]
y_val = y_train[val_indices]
seqid_val = seqid_train_used[val_indices]

print("Subtrain size:", len(X_subtrain_tok))
print("Val size:", len(X_val_tok))
print("Test size:", len(X_test_tok))

class TokenSequenceDataset(Dataset):
    def __init__(self, X_tokens, lengths, y, seqids):
        self.X_tokens = torch.tensor(X_tokens, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
        self.seqids = np.array(seqids)

        assert len(self.X_tokens) == len(self.lengths) == len(self.y) == len(self.seqids)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "input_ids": self.X_tokens[idx],
            "length": self.lengths[idx],
            "label": self.y[idx],
            "seqid": self.seqids[idx],
        }

train_dataset = TokenSequenceDataset(X_subtrain_tok, subtrain_lens, y_subtrain, seqid_subtrain)
val_dataset = TokenSequenceDataset(X_val_tok, val_lens, y_val, seqid_val)
test_dataset = TokenSequenceDataset(X_test_tok, test_lens, y_test, seqid_test_used)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Subtrain size: 5159
Val size: 911
Test size: 1518
Train batches: 81
Val batches: 15
Test batches: 24


Cell 12: Define Token-LSTM model

In [12]:
# Cell 12: Define Token-LSTM model

class TokenLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        num_layers: int = 1,
        dropout: float = 0.2,
        pad_token_id: int = 0,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_token_id,
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, lengths):
        emb = self.embedding(input_ids)  # (B, L, E)

        packed = nn.utils.rnn.pack_padded_sequence(
            emb,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, (h_n, _) = self.lstm(packed)
        last_hidden = h_n[-1]  # (B, hidden_dim)

        out = self.dropout(last_hidden)
        logits = self.classifier(out)  # (B, num_classes)
        return logits


EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_CLASSES = len(classes)

model = TokenLSTMClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    num_layers=1,
    dropout=0.2,
    pad_token_id=PAD_TOKEN_ID,
).to(DEVICE)

print(model)

TokenLSTMClassifier(
  (embedding): Embedding(4097, 64, padding_idx=0)
  (lstm): LSTM(64, 128, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=128, out_features=18, bias=True)
)


Cell 13: Training setup

In [13]:
# Cell 13: Training setup

LEARNING_RATE = 1e-3
#EPOCHS = 15
EPOCHS = 30

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("LEARNING_RATE:", LEARNING_RATE)
print("EPOCHS:", EPOCHS)

LEARNING_RATE: 0.001
EPOCHS: 30


Cell 14: Define BFRB mapping + binary threshold helpers

In [14]:
# Cell 14: Define BFRB mapping + binary threshold helpers

BFRB_CLASS_NAMES = {
    "Above ear - pull hair",
    "Cheek - pinch skin",
    "Eyebrow - pull hair",
    "Eyelash - pull hair",
    "Forehead - pull hairline",
    "Forehead - scratch",
    "Neck - pinch skin",
    "Neck - scratch",
}

bfrb_class_ids = sorted([label2id[name] for name in BFRB_CLASS_NAMES if name in label2id])

print("BFRB class ids:", bfrb_class_ids)
print("BFRB class names:", [id2label[i] for i in bfrb_class_ids])

def to_binary_labels(y_multiclass, positive_ids):
    """
    18-class -> binary
    BFRB = 1, non-BFRB = 0
    """
    return np.array([1 if y in positive_ids else 0 for y in y_multiclass], dtype=np.int64)

def to_9class_labels(y_multiclass, bfrb_ids):
    """
    18-class -> 9-class (paper setting)
    - 8 BFRB classes keep their original ids
    - all non-BFRB classes collapse into one shared class id = 8
    """
    bfrb_ids = list(sorted(bfrb_ids))
    bfrb_id_to_9class = {orig_id: new_id for new_id, orig_id in enumerate(bfrb_ids)}
    non_bfrb_class_id = len(bfrb_ids)  # should be 8

    y_new = []
    for y in y_multiclass:
        if y in bfrb_id_to_9class:
            y_new.append(bfrb_id_to_9class[y])
        else:
            y_new.append(non_bfrb_class_id)

    return np.array(y_new, dtype=np.int64)

def binary_probs_from_logits(logits: np.ndarray, positive_ids):
    """
    logits: (N, C)
    return:
      binary_probs: (N,) = sum of softmax probs over BFRB classes
    """
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    return probs[:, positive_ids].sum(axis=1)

def find_best_binary_threshold(binary_targets, binary_probs, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.951, 0.01)

    best_threshold = 0.50
    best_f1 = -1.0

    for thr in thresholds:
        preds = (binary_probs >= thr).astype(np.int64)
        f1 = f1_score(binary_targets, preds, average="binary")

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = float(thr)

    return best_threshold, best_f1

# quick sanity check
sample_y = np.array([0, 1, 2, 10, 15])
print("sample binary:", to_binary_labels(sample_y, bfrb_class_ids))
print("sample 9-class:", to_9class_labels(sample_y, bfrb_class_ids))

BFRB class ids: [0, 1, 3, 4, 6, 7, 9, 10]
BFRB class names: ['Above ear - pull hair', 'Cheek - pinch skin', 'Eyebrow - pull hair', 'Eyelash - pull hair', 'Forehead - pull hairline', 'Forehead - scratch', 'Neck - pinch skin', 'Neck - scratch']
sample binary: [1 1 0 1 0]
sample 9-class: [0 1 8 7 8]


Cell 15: Train one epoch

In [15]:
# Cell 15: Train one epoch

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_samples = 0

    all_preds = []
    all_targets = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["length"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        preds = torch.argmax(logits, dim=1)

        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(labels.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    avg_loss = total_loss / total_samples

    # ===== paper-style 9-class macro F1 =====
    y_true_9 = to_9class_labels(all_targets, bfrb_class_ids)
    y_pred_9 = to_9class_labels(all_preds, bfrb_class_ids)
    macro_f1_9class = f1_score(y_true_9, y_pred_9, average="macro")

    # ===== binary F1 =====
    binary_targets = to_binary_labels(all_targets, bfrb_class_ids)
    binary_preds = to_binary_labels(all_preds, bfrb_class_ids)
    binary_f1 = f1_score(binary_targets, binary_preds, average="binary")

    # 可选：18-class macro F1（附加分析，不作为论文主指标）
    macro_f1_18class = f1_score(all_targets, all_preds, average="macro")

    return avg_loss, macro_f1_9class, binary_f1, macro_f1_18class

Cell 16: Evaluate model

In [16]:
# Cell 16: Evaluate model

def evaluate(model, loader, criterion, device, binary_threshold=None, tune_binary_threshold=False):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_logits = []
    all_preds = []
    all_targets = []
    all_seqids = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["length"].to(device)
            labels = batch["label"].to(device)
            seqids = batch["seqid"]

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = torch.argmax(logits, dim=1)

            all_logits.append(logits.detach().cpu().numpy())
            all_preds.append(preds.detach().cpu().numpy())
            all_targets.append(labels.detach().cpu().numpy())
            all_seqids.extend(list(seqids))

    all_logits = np.concatenate(all_logits, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    avg_loss = total_loss / total_samples

    # ===== paper-style 9-class macro F1 =====
    y_true_9 = to_9class_labels(all_targets, bfrb_class_ids)
    y_pred_9 = to_9class_labels(all_preds, bfrb_class_ids)
    macro_f1_9class = f1_score(y_true_9, y_pred_9, average="macro")

    # ===== binary F1 from argmax mapping (reference only) =====
    binary_targets = to_binary_labels(all_targets, bfrb_class_ids)
    binary_preds_argmax = to_binary_labels(all_preds, bfrb_class_ids)
    binary_f1_argmax = f1_score(binary_targets, binary_preds_argmax, average="binary")

    # ===== binary F1 from aggregated BFRB probabilities + threshold =====
    binary_probs = binary_probs_from_logits(all_logits, bfrb_class_ids)

    if tune_binary_threshold:
        binary_threshold, binary_f1 = find_best_binary_threshold(binary_targets, binary_probs)
    else:
        if binary_threshold is None:
            binary_threshold = 0.50
        binary_preds = (binary_probs >= binary_threshold).astype(np.int64)
        binary_f1 = f1_score(binary_targets, binary_preds, average="binary")

    # ===== optional: 18-class macro F1 =====
    macro_f1_18class = f1_score(all_targets, all_preds, average="macro")

    return {
        "loss": avg_loss,
        "macro_f1_9class": macro_f1_9class,
        "binary_f1": binary_f1,
        "binary_f1_argmax": binary_f1_argmax,
        "binary_threshold": float(binary_threshold),
        "binary_probs": binary_probs,
        "macro_f1_18class": macro_f1_18class,
        "logits": all_logits,
        "preds": all_preds,
        "targets": all_targets,
        "seqids": np.array(all_seqids, dtype=object),
    }

Cell 17: Train loop (validation threshold tuning + binary-aware best model selection)

In [17]:
# Cell 17: Train loop (validation threshold tuning + binary-aware best model selection)

history = []

best_val_binary_f1 = -1.0
best_binary_threshold = 0.50
best_state_dict = None
best_val_eval = None
best_test_eval = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_macro_f1_9class, train_binary_f1, train_macro_f1_18class = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )

    val_result = evaluate(
        model,
        val_loader,
        criterion,
        DEVICE,
        tune_binary_threshold=True,
    )

    test_result = evaluate(
        model,
        test_loader,
        criterion,
        DEVICE,
        binary_threshold=val_result["binary_threshold"],
        tune_binary_threshold=False,
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_macro_f1_9class": train_macro_f1_9class,
        "train_binary_f1_argmax": train_binary_f1,
        "train_macro_f1_18class": train_macro_f1_18class,
        "val_loss": val_result["loss"],
        "val_macro_f1_9class": val_result["macro_f1_9class"],
        "val_binary_f1": val_result["binary_f1"],
        "val_binary_f1_argmax": val_result["binary_f1_argmax"],
        "val_binary_threshold": val_result["binary_threshold"],
        "val_macro_f1_18class": val_result["macro_f1_18class"],
        "test_loss": test_result["loss"],
        "test_macro_f1_9class": test_result["macro_f1_9class"],
        "test_binary_f1": test_result["binary_f1"],
        "test_binary_f1_argmax": test_result["binary_f1_argmax"],
        "test_binary_threshold": test_result["binary_threshold"],
        "test_macro_f1_18class": test_result["macro_f1_18class"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"train_binary_f1_argmax={train_binary_f1:.4f} | "
        f"val_binary_f1={val_result['binary_f1']:.4f} @thr={val_result['binary_threshold']:.2f} | "
        f"test_binary_f1={test_result['binary_f1']:.4f} | "
        f"val_macro_f1_9class={val_result['macro_f1_9class']:.4f} | "
        f"test_macro_f1_9class={test_result['macro_f1_9class']:.4f}"
    )

    if val_result["binary_f1"] > best_val_binary_f1:
        best_val_binary_f1 = val_result["binary_f1"]
        best_binary_threshold = val_result["binary_threshold"]
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_val_eval = val_result
        best_test_eval = test_result

Epoch 30 | train_loss=0.7317 | train_binary_f1_argmax=0.9496 | val_binary_f1=0.8437 @thr=0.21 | test_binary_f1=0.8376 | val_macro_f1_9class=0.2726 | test_macro_f1_9class=0.2922


Cell 18: Load best model and finalize evaluation

In [18]:
# Cell 18: Load best model and finalize evaluation

assert best_state_dict is not None, "No best model was saved."

model.load_state_dict(best_state_dict)

final_val_eval = evaluate(
    model,
    val_loader,
    criterion,
    DEVICE,
    tune_binary_threshold=True,
)

final_eval = evaluate(
    model,
    test_loader,
    criterion,
    DEVICE,
    binary_threshold=best_binary_threshold,
    tune_binary_threshold=False,
)

print("\nFinal validation results (best checkpoint):")
print("val_loss:", final_val_eval["loss"])
print("val_macro_f1_9class:", final_val_eval["macro_f1_9class"])
print("val_binary_f1:", final_val_eval["binary_f1"])
print("val_binary_f1_argmax:", final_val_eval["binary_f1_argmax"])
print("val_macro_f1_18class:", final_val_eval["macro_f1_18class"])
print("best_binary_threshold:", best_binary_threshold)

print("\nFinal test results (best model):")
print("test_loss:", final_eval["loss"])
print("test_macro_f1_9class:", final_eval["macro_f1_9class"])
print("test_binary_f1:", final_eval["binary_f1"])
print("test_binary_f1_argmax:", final_eval["binary_f1_argmax"])
print("test_macro_f1_18class:", final_eval["macro_f1_18class"])
print("binary_threshold_used:", final_eval["binary_threshold"])
print("logits shape:", final_eval["logits"].shape)
print("targets shape:", final_eval["targets"].shape)
print("seqids shape:", final_eval["seqids"].shape)


Final validation results (best checkpoint):
val_loss: 2.2651491481831254
val_macro_f1_9class: 0.28085907263011983
val_binary_f1: 0.860248447204969
val_binary_f1_argmax: 0.8226351351351351
val_macro_f1_18class: 0.1999507156919586
best_binary_threshold: 0.42000000000000004

Final test results (best model):
test_loss: 2.2163157642123257
test_macro_f1_9class: 0.2856092499391873
test_binary_f1: 0.847457627118644
test_binary_f1_argmax: 0.8180435884439939
test_macro_f1_18class: 0.20601973666695805
binary_threshold_used: 0.42000000000000004
logits shape: (1518, 18)
targets shape: (1518,)
seqids shape: (1518,)


Cell 19: output some files

In [19]:
# Cell 19: output some files

import os
import json
import numpy as np

from sklearn.metrics import accuracy_score
acc = accuracy_score(final_eval["targets"], final_eval["preds"])

# the root dir of this project
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

# the dir for the output files
MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "results",
    "Model11_Training"
)

os.makedirs(MODEL_DIR, exist_ok=True)

# the paths for the files
OUT_LOGITS = os.path.join(MODEL_DIR, "Model_11_TOF_Only_Token_LSTM_test_logits.npy")
OUT_Y = os.path.join(MODEL_DIR, "Model_11_TOF_Only_Token_LSTM_test_y.npy")
OUT_SEQ_IDS = os.path.join(MODEL_DIR, "Model_11_TOF_Only_Token_LSTM_test_seqids.npy")
OUT_METRICS = os.path.join(MODEL_DIR, "model11_training_summary.json")

# save of these files
np.save(OUT_LOGITS, final_eval["logits"])
np.save(OUT_Y, final_eval["targets"])
np.save(OUT_SEQ_IDS, final_eval["seqids"])

metrics_to_save = {

    "final_metrics": {
        "acc": float(acc),
        "macroF1": float(final_eval["macro_f1_9class"]),
        "binaryF1": float(final_eval["binary_f1"])
    }
}

with open(OUT_METRICS, "w") as f:
    json.dump(metrics_to_save, f, indent=4)


print("Saved to:", MODEL_DIR)

Saved to: D:\0-about me\2-FDU\4-CSCI 6806 CAPSTONE\1_from github\2026-winter-capstone-project-2026winter-capstone-group-11\results\Model11_Training


Cell 20: Training history

In [20]:
# Cell 20: Training history

history_df = pd.DataFrame(history)
display(history_df)

,epoch,train_loss,train_macro_f1_9class,train_binary_f1_argmax,train_macro_f1_18class,val_loss,val_macro_f1_9class,val_binary_f1,val_binary_f1_argmax,val_binary_threshold,val_macro_f1_18class,test_loss,test_macro_f1_9class,test_binary_f1,test_binary_f1_argmax,test_binary_threshold,test_macro_f1_18class
0,1,2.600651,0.183783,0.714588,0.103979,2.350338,0.198361,0.835347,0.813531,0.40,0.108555,2.332675,0.212959,0.830601,0.815816,0.40,0.135951
1,2,2.258041,0.252218,0.819357,0.153364,2.276609,0.222674,0.844749,0.825581,0.35,0.129728,2.237325,0.244089,0.837188,0.818874,0.35,0.143987
2,3,2.169723,0.279514,0.826417,0.179402,2.233475,0.253998,0.847962,0.840486,0.46,0.144456,2.205070,0.273713,0.839447,0.823762,0.46,0.173626
3,4,2.114928,0.302497,0.823883,0.207741,2.219550,0.233395,0.852784,0.816638,0.37,0.149291,2.197286,0.271851,0.842937,0.815424,0.37,0.184753
4,5,2.059438,0.311988,0.833309,0.225138,2.206683,0.240119,0.850965,0.829876,0.39,0.160192,2.185847,0.273411,0.843735,0.821752,0.39,0.182959
5,6,2.007023,0.334265,0.837662,0.247038,2.207148,0.254179,0.849421,0.835796,0.44,0.170575,2.174197,0.287313,0.845574,0.824000,0.44,0.186844
6,7,1.955733,0.356321,0.845178,0.262653,2.233250,0.256334,0.847537,0.835526,0.49,0.160179,2.181350,0.298814,0.836520,0.821159,0.49,0.206005
7,8,1.906111,0.361573,0.845083,0.280834,2.224240,0.255404,0.850837,0.834580,0.37,0.177762,2.170768,0.296421,0.847489,0.824482,0.37,0.217935
8,9,1.857485,0.384265,0.845487,0.308161,2.239937,0.281892,0.851227,0.830313,0.46,0.195041,2.186133,0.307766,0.842155,0.823881,0.46,0.203360
9,10,1.813075,0.402966,0.850045,0.328174,2.243945,0.275114,0.849042,0.834154,0.39,0.192543,2.175650,0.308765,0.843823,0.827586,0.39,0.223297


Cell 21: save training curves for Model 11

In [21]:
# Cell 21: save training curves for Model 11

import os
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("default")


PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))


PLOT_DIR = os.path.join(PROJECT_ROOT, "plots", "model_11_training_curves")
os.makedirs(PLOT_DIR, exist_ok=True)

# ===== history -> DataFrame =====
history_df = pd.DataFrame(history)

print("history_df columns:", history_df.columns.tolist())
print("Saving plots to:", PLOT_DIR)


# 1) loss curve
plt.figure(figsize=(8, 5))
if "train_loss" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
if "val_loss" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
if "test_loss" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["test_loss"], label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model 11 Loss Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "loss_curve.png"), dpi=200)
plt.close()


# 2) binary f1 curve
plt.figure(figsize=(8, 5))
if "train_binary_f1_argmax" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["train_binary_f1_argmax"], label="Train Binary F1 (Argmax)")
if "val_binary_f1" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["val_binary_f1"], label="Val Binary F1")
if "test_binary_f1" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["test_binary_f1"], label="Test Binary F1")

plt.xlabel("Epoch")
plt.ylabel("Binary F1")
plt.title("Model 11 Binary F1 Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "binary_f1_curve.png"), dpi=200)
plt.close()


# 3) macro f1 curve
plt.figure(figsize=(8, 5))
if "train_macro_f1_9class" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["train_macro_f1_9class"], label="Train Macro F1 (9-class)")
if "val_macro_f1_9class" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["val_macro_f1_9class"], label="Val Macro F1 (9-class)")
if "test_macro_f1_9class" in history_df.columns:
    plt.plot(history_df["epoch"], history_df["test_macro_f1_9class"], label="Test Macro F1 (9-class)")


plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Model 11 Macro F1 Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "macro_f1_curve.png"), dpi=200)
plt.close()

print("Saved:", os.path.join(PLOT_DIR, "loss_curve.png"))
print("Saved:", os.path.join(PLOT_DIR, "binary_f1_curve.png"))
print("Saved:", os.path.join(PLOT_DIR, "macro_f1_curve.png"))

Saved: D:\0-about me\2-FDU\4-CSCI 6806 CAPSTONE\1_from github\2026-winter-capstone-project-2026winter-capstone-group-11\plots\model_11_training_curves\loss_curve.png
Saved: D:\0-about me\2-FDU\4-CSCI 6806 CAPSTONE\1_from github\2026-winter-capstone-project-2026winter-capstone-group-11\plots\model_11_training_curves\binary_f1_curve.png
Saved: D:\0-about me\2-FDU\4-CSCI 6806 CAPSTONE\1_from github\2026-winter-capstone-project-2026winter-capstone-group-11\plots\model_11_training_curves\macro_f1_curve.png
